# Extract Meta Data

In [1]:
from pathlib import Path

import os
import sys

# --- Paths / imports -------------------------------------------------
PROJECT_ROOT = Path.cwd().parent
PREPROCESSING_DIR = PROJECT_ROOT / "functions" / "preprocessing"
for p in (PROJECT_ROOT, PREPROCESSING_DIR):
    if str(p) not in sys.path:
        sys.path.append(str(p))

from server_config import datapath, proj_sheet, preprocessed_path, raw_path, backup_path

import glob
import pickle
from IPython.display import Markdown
from server_config import datapath, redcap_path, preprocessed_path

import pandas as pd
import numpy as np
import datetime as dt



In [2]:
with open(preprocessed_path + '/ema_content.pkl', 'rb') as file:
    df_ema_content = pickle.load(file)  

df_monitoring = pd.read_csv(f"https://docs.google.com/spreadsheets/d/{proj_sheet}/export?format=csv")
    
df_redcap_final = pd.read_spss(redcap_path + "/sp1_cleaned" + "/baseline_raw_20251012.sav")

# Anpassen an aktuelle Daten 
df_redcap_process = pd.read_csv(redcap_path + "/verlauf" + "/FOR_Verlauf.csv")
df_redcap_process_zert = pd.read_csv(redcap_path + "/verlauf" + "/FOR_Zertifizierung_Verlauf.csv")
df_redcap_t5 = pd.read_csv(redcap_path + "/t5" + "/T5_BDI_08012026.csv") 

df_ema_onboarding = pd.read_csv(redcap_path + "/ema_onboarding" + "/EMA_Onboarding_17062026.csv")
df_ema_onboarding_zert = pd.read_csv(redcap_path + "/ema_onboarding" + "/ZERT_EMA_Onboarding_17062026.csv")


In [3]:
df_redcap_final_short = df_redcap_final[["for_id","basic_documentation_sheet_timestamp", "ema_start_date" ]]

### 1. Monitoring

In [4]:
df_monitoring = df_monitoring.copy()

df_monitoring.rename(columns = {"Pseudonym": "id",
                                "FOR_ID": "for_id",
                                "EMA_ID": "ema_id", 
                                "Status": "study_status",
                                "Studienversion":"study_version", 
                                "Start EMA Baseline": "ema_base_start", 
                                "Ende EMA Baseline": "ema_base_end", 
                                "Freischaltung/ Start EMA T20": "ema_t20_start",
                                "Ende EMA T20":"ema_t20_end", 
                                "Freischaltung/ Start EMA Post":"ema_post_start",
                               "Ende EMA Post":"ema_post_end", 
                                "T20=Post":"t20_post",
                               'Telefonat stattgefunden?': 'ema_phonecall_happened',
                               "Status reduziert":"study_status_short",
                               'Dropout vor KV': "dropout_pre_14",
                                'Dropout nach KV':"dropout_post_14",
                                'Dropout Grund (frei)': "dropout_reason_text"
                               }, 
                     inplace=True)

df_monitoring = df_monitoring[['for_id', 'ema_id', 'id', 'study_version', 'study_status',
       't20_post', 'ema_base_start', 'ema_base_end', 'ema_t20_start', 'ema_t20_end',
       'ema_post_start', 'ema_post_end', "ema_phonecall_happened","study_status_short", 'dropout_pre_14', 'dropout_post_14']]

df_monitoring["id"] = df_monitoring["id"].str[:4]
df_monitoring["for_id"] = df_monitoring.for_id.str.strip()

df_monitoring["ema_base_start"] = pd.to_datetime(
    df_monitoring["ema_base_start"], dayfirst=True, errors="coerce", utc=True
)
df_monitoring["ema_base_end"] = pd.to_datetime(
    df_monitoring["ema_base_end"], dayfirst=True, errors="coerce", utc=True
)

In [5]:
meta_cols_monitoring = ['for_id', 'id', 'study_version','t20_post', 'ema_base_start', 'ema_base_end', 'ema_t20_start', 'ema_t20_end',
       'ema_post_start', 'ema_post_end', "ema_phonecall_happened","study_status_short", 'dropout_pre_14', 'dropout_post_14']

In [6]:
df_monitoring_meta = df_monitoring[meta_cols_monitoring]

In [7]:
df_monitoring_short = df_monitoring[["for_id", "study_version"]]

### 2. Duration until T5, T20, Post 

- 13023 T5 FB erst nach T10 ausgefüllt
- 11011: Für die T5 (03.01.2024) wurde das Datum der T6 (16.01.2024) eingetragen. Das erklärt aber nicht den langen Abstand zwischen EMAstart/IC (21.06.2023) und T5. Tessa plant, morgen bei Epikur mögliche Erklärungen dafür zu prüfen (Sitzungsabstände und Notizen).
- 11012: Die Daten für EMAstart/IC/T5 sind korrekt. Laut der Verlaufsdokumentation gab es Schwierigkeiten bei der Terminfindung mit zwei Therapeut*innen. Das könnte den langen Abstand zwischen dem IC-Termin (21.06.2023) und der T5 (04.03.2024) erklären.
- 11063: Für die T5-Messung wurde ein falsches Datum eingetragen. Das korrekte Datum ist der 31.10.2024. Der Abstand zwischen der T5 und der T5-Messung lässt sich dadurch erklären, dass die Fragebögen zu T7 ausgefüllt wurden (siehe Verlaufsdoku).
- FOR14137 war ein Tippfehler (ist 2024 statt 2025), das habe ich in RedCap korrigiert.
- FOR14138 hier sind die Daten korrekt, die Person hatte lange keine Sitzungen und ist dann auch zum Drop-Out geworden. Die T5 Messungen war mehrere Monate nach T5, allerdings haben dann auch keine Sitzungen mehr stattgefunden, da Abbruch.

In [8]:
col_list = ['for_id','t5_session_date','t1_session_date','t20_session_date','ic_date','t5_assessment_date',
 't20_assessment_date','post_assessment_date','dropout_date', 'ema_date']

In [9]:
df_redcap_process_short = df_redcap_process[col_list]
df_redcap_process_zert_short = df_redcap_process_zert[col_list]
df_redcap_process_short = pd.concat([df_redcap_process_short, df_redcap_process_zert_short])
df_redcap_process_short.dropna(subset= "ema_date", inplace=True)

In [10]:
df_redcap_process_short = df_redcap_process_short.merge(df_monitoring_short, on= "for_id", how="left")

In [11]:
df_redcap_process_short = df_redcap_process_short.loc[df_redcap_process_short.study_version.isin(["Lang","Lang (Wechsel)"])]

In [12]:

df = df_redcap_process_short.copy()  # avoid SettingWithCopy + work on a real DF

start_col = "ema_date"
targets = [
    "t1_session_date", "t5_session_date", "t20_session_date",
    "t5_assessment_date", "t20_assessment_date", "post_assessment_date",
    "dropout_date",
]
date_cols = [start_col] + targets

# 1) Ensure all are real datetimes (NA/invalid -> NaT)
for c in date_cols:
    df.loc[:, c] = pd.to_datetime(df[c], errors="coerce")
    
# 2) Correct date for ID FOR13053
idx = df.index[df["for_id"].eq("FOR13053")][0]          # first matching row
df.loc[idx, "ema_date"] = df.loc[idx, "ema_date"] - pd.DateOffset(years=1)


# If you want start per for_id (in case ic_date varies within id), use this instead:
# start = df.groupby("for_id")[start_col].transform("min")
start = df[start_col]

# 3) Durations (NaT anywhere -> NaN)
for c in targets:
    delta = df[c].sub(start)

    # This line makes the code robust if delta is "object" (mixed datetime/date/timedelta)
    delta = pd.to_timedelta(delta, errors="coerce")

    df.loc[:, f"duration_to_{c}"] = delta.dt.days.astype("float")


In [13]:
meta_cols_duration = ['for_id', 'duration_to_t1_session_date', 'duration_to_t5_session_date',
       'duration_to_t20_session_date', 'duration_to_t5_assessment_date',
       'duration_to_t20_assessment_date', 'duration_to_post_assessment_date',
       'duration_to_dropout_date']

In [14]:
df_duration_meta = df[meta_cols_duration]

### 3. EMA Onboarding 

In [15]:
df_ema_onboarding_short = df_ema_onboarding[['for_id','ema_smartphone','ema_wear_exp','ema_sleep','ema_special_event','ema_phonecall_1','ema_phonecall',
                                             'ema_phonecall_2', 'ema_phonecall_3','ema_phonecall_4','ema_phonecall_5',  
                                            'ema_cancelled_contact','ema_cancelled_reason___1','ema_cancelled_reason___2','ema_cancelled_reason___3',
                                             'ema_cancelled_reason___4','ema_cancelled_reason___5', 'ema_cancelled_reason___6','ema_cancelled_reason___7',
                                             'ema_cancelled_reason___8','ema_cancelled_reason___9',  'ema_cancelled','ema_cancelled_date' ]]


In [16]:
df_ema_onboarding_zert_short = df_ema_onboarding_zert[['for_id','ema_smartphone','ema_wear_exp','ema_sleep','ema_special_event','ema_phonecall',
                                                       'ema_phonecall_1','ema_phonecall_2', 'ema_phonecall_3','ema_phonecall_4','ema_phonecall_5',  
                                            'ema_cancelled_contact','ema_cancelled_reason___1','ema_cancelled_reason___2','ema_cancelled_reason___3',
                                             'ema_cancelled_reason___4','ema_cancelled_reason___5', 'ema_cancelled_reason___6','ema_cancelled_reason___7',
                                             'ema_cancelled_reason___8','ema_cancelled_reason___9',  'ema_cancelled','ema_cancelled_date']]


In [17]:
df_onboarding_final = pd.concat([df_ema_onboarding_zert_short, df_ema_onboarding_short], ignore_index=True)


In [18]:
df_onboarding_final = df_onboarding_final.merge(df_monitoring_short, on="for_id", how= "right")

In [19]:
df_onboarding_final = (df_onboarding_final.groupby('for_id', as_index=False).first())  # takes first non-NaN value per column

In [20]:
meta_cols_redcap = ['for_id', 'ema_smartphone', 'ema_wear_exp', 'ema_sleep','ema_special_event', 'ema_phonecall_1','ema_phonecall',
       'ema_phonecall_2', 'ema_phonecall_3', 'ema_phonecall_4','ema_phonecall_5']

In [21]:
df_onboarding_meta = df_onboarding_final[meta_cols_redcap]

### 4. Final Merge

In [22]:
df_meta = df_onboarding_meta.merge(df_duration_meta, on = "for_id", how="left")

In [23]:
df_meta = df_meta.merge(df_monitoring_meta, on = "for_id", how= "right")

In [24]:
df_meta.to_csv(preprocessed_path + "/meta/"+ "SP6_meta_data.csv", index=False)